# CosyVoice 2 · 개인 코랩 v2.9.2

**런타임 → 런타임 유형 변경 → T4 GPU**를 선택하고, **1번 왼쪽 ▶**를 누르세요.
`준비 완료`가 나오면 사이트 연결은 **4번**, 코랩 안에서 직접 만들기는 **2·3번**을 사용하세요.

설치는 처음에 시간이 걸립니다. 모델 준비 단계와 실제 로딩 기록을 바로 표시합니다.
모델 준비가 10분 안에 끝나지 않으면 중단하고 마지막 기록을 표시합니다.
2분 이상 걸릴 때 나오는 `Timeout`과 실행 위치 기록은 멈춘 지점을 찾는 자료이며, 실제 중단 여부는 마지막 완료·오류 문구로 확인하세요.
같은 런타임에서 다시 실행하면 기존 설치와 모델 다운로드를 재사용합니다.
참조 음성은 배경음 없이 한 사람만 말하는 **3~30초 WAV/MP3(10MB 이하)**와 정확한 실제 대사를 준비하세요.

**스타일 지원 업데이트:** 기존 서버가 실행 중이면 **런타임 → 세션 다시 시작** 후 1번을 실행하세요. 사이트에서 기본 이외의 스타일을 고르면 연기 지시를 전달합니다. 나이·음색은 참조 목소리의 영향을 받습니다.


In [ ]:
#@title 1. CosyVoice 2 설치 및 실행 · v2.9.2 · 모델 준비 단계·실제 로그 표시
from pathlib import Path
from collections import deque
import subprocess
import sys
server_source = (
    '# -*- coding: utf-8 -*-\n'
    '"""Self-contained CosyVoice 2 Colab runner. No edits to upstream model code.\n'
    '\n'
    '--setup: install an isolated Python 3.10 environment and start the local API.\n'
    '--tunnel: optionally expose the ready API to AI Voice Studio.\n'
    '--serve: internal worker, launched with the isolated Python interpreter.\n'
    '"""\n'
    'import argparse\n'
    'import hashlib\n'
    'import io\n'
    'import json\n'
    'import logging\n'
    'import os\n'
    'from pathlib import Path\n'
    'import re\n'
    'import secrets\n'
    'import shutil\n'
    'import socket\n'
    'import subprocess\n'
    'import sys\n'
    'import tempfile\n'
    'import threading\n'
    'import time\n'
    'import traceback\n'
    'import urllib.request\n'
    '\n'
    "SOURCE_REVISION = '074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc'\n"
    "COSY_MODEL_REVISION = 'eec1ae6c79877dbd9379285cf8789c9e0879293d'\n"
    '\n'
    '# Retain the existing installation/cache path so an interrupted install can resume.\n'
    "ENGINE = 'cosyvoice'\n"
    "LABEL = 'CosyVoice 2'\n"
    "ROOT = Path('/content/ai_voice_dual_v1/cosyvoice')\n"
    "SOURCE = ROOT / 'CosyVoice'\n"
    "PYTHON = ROOT / 'venv/bin/python'\n"
    "MODEL = SOURCE / 'pretrained_models/CosyVoice2-0.5B'\n"
    "STATE = ROOT / 'state.json'\n"
    "SERVICE = 'ai-voice-studio-cosyvoice'\n"
    "SERVER_VERSION = '2.9.2'\n"
    'MODEL_REVISION = COSY_MODEL_REVISION\n'
    '\n'
    '\n'
    'def run(args, label):\n'
    "    print('\\n▶ ' + label, flush=True)\n"
    "    with (ROOT / 'setup.log').open('a', encoding='utf-8') as log:\n"
    "        log.write('\\n▶ ' + label + '\\n')\n"
    '        log.flush()\n'
    '        proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE,\n'
    '                                stderr=subprocess.STDOUT, text=True, bufsize=1)\n'
    '        try:\n'
    '            for line in proc.stdout:\n'
    "                print(line, end='', flush=True)\n"
    '                log.write(line)\n'
    '                log.flush()\n'
    '            code = proc.wait()\n'
    '        except BaseException:\n'
    '            proc.terminate()\n'
    '            raise\n'
    '    if code:\n'
    "        raise RuntimeError(f'{label} 실패 (종료 코드 {code}). 바로 위 오류를 확인해주세요.')\n"
    '\n'
    '\n'
    'def read_state():\n'
    '    try:\n'
    "        return json.loads(STATE.read_text(encoding='utf-8'))\n"
    '    except (OSError, ValueError):\n'
    '        return {}\n'
    '\n'
    '\n'
    'def health(base, require_style=True):\n'
    "    with urllib.request.urlopen(base + '/health', timeout=5) as response:\n"
    '        data = json.load(response)\n'
    "    return (data.get('service') == SERVICE and data.get('ready') is True\n"
    "            and (not require_style or 'style_instruction' in data.get('capabilities', [])))\n"
    '\n'
    '\n'
    'def setup():\n'
    "    if not Path('/content').is_dir():\n"
    "        raise RuntimeError('이 파일은 Google Colab에서 실행해주세요.')\n"
    '    ROOT.mkdir(parents=True, exist_ok=True)\n'
    '    state = read_state()\n'
    '    existing_ready = False\n'
    '    try:\n'
    "        existing_ready = bool(state.get('base')) and health(state['base'], require_style=False)\n"
    '    except Exception:\n'
    '        pass\n'
    '    if existing_ready:\n'
    "        if health(state['base']):\n"
    "            print(f'✅ {LABEL} v{SERVER_VERSION} 준비 완료. 아래 음성 생성 셀을 실행하세요.', flush=True)\n"
    '            return\n'
    "        raise RuntimeError('이전 CosyVoice 서버가 실행 중입니다. 런타임 → 세션 다시 시작 후 이 노트북의 1번을 실행해주세요. 설치한 모델은 같은 런타임에 유지됩니다.')\n"
    "    if shutil.which('nvidia-smi') is None:\n"
    "        raise RuntimeError('GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU를 선택해주세요.')\n"
    "    run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 'GPU 확인')\n"
    "    run(['apt-get', 'update', '-qq'], '시스템 패키지 목록 갱신')\n"
    "    run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'sox', 'libsox-dev',\n"
    "         'libsndfile1', 'build-essential', 'git'], '오디오 도구 설치')\n"
    '    if not PYTHON.is_file():\n'
    "        run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'uv'], 'Python 환경 도구 설치')\n"
    "        run([sys.executable, '-m', 'uv', 'python', 'install', '3.10'], '별도 Python 3.10 설치')\n"
    "        run([sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', ROOT / 'venv'], '독립 실행 환경 생성')\n"
    "    if not (SOURCE / '.git').is_dir():\n"
    "        run(['git', 'clone', '--filter=blob:none', 'https://github.com/FunAudioLLM/CosyVoice.git', SOURCE], '공식 CosyVoice 소스 다운로드')\n"
    "    run(['git', '-C', SOURCE, 'checkout', '--detach', SOURCE_REVISION], '확인한 소스 버전 선택')\n"
    "    run(['git', '-C', SOURCE, 'submodule', 'update', '--init', '--recursive'], 'Matcha-TTS 소스 준비')\n"
    '    # Keep upstream versions except the documented inference-only changes below.\n'
    '    requirements = []\n'
    "    for line in (SOURCE / 'requirements.txt').read_text().splitlines():\n"
    "        if line.startswith(('deepspeed', 'tensorrt', 'gradio', 'fastapi-cli')):\n"
    '            continue  # optional acceleration/training/UI; this server uses none of these\n'
    "        if line.startswith('diffusers=='):\n"
    "            line = 'diffusers==0.32.2'  # compatible with modern huggingface_hub (no cached_download import)\n"
    '        requirements.append(line)\n'
    "    requirements += ['huggingface-hub==0.30.2', 'python-multipart>=0.0.18,<0.1', 'setuptools<81']\n"
    "    req_file = ROOT / 'inference-requirements.txt'\n"
    "    req_file.write_text('\\n'.join(requirements) + '\\n')\n"
    "    # Whisper's pinned source distribution imports pkg_resources while building.\n"
    "    # The runtime setuptools pin alone does not apply inside pip's isolated build.\n"
    "    build_constraints = ROOT / 'build-constraints.txt'\n"
    "    build_constraints.write_text('setuptools<81\\n')\n"
    '    stamp = hashlib.sha256(req_file.read_bytes() + build_constraints.read_bytes()).hexdigest()\n'
    "    marker = ROOT / 'dependencies.ok'\n"
    '    if not marker.exists() or marker.read_text() != stamp:\n'
    "        run([PYTHON, '-m', 'pip', 'install', 'pip>=25.3,<26', 'setuptools<81', 'wheel', 'Cython<4'], '설치 도구 준비')\n"
    "        # Install the matching CUDA pair first; do not touch Colab's own Torch.\n"
    "        run([PYTHON, '-m', 'pip', 'install', 'torch==2.3.1', 'torchaudio==2.3.1',\n"
    "             '--index-url', 'https://download.pytorch.org/whl/cu121'], 'GPU용 Torch 및 오디오 패키지 설치')\n"
    "        run([PYTHON, '-m', 'pip', 'install', '-r', req_file,\n"
    "             '--build-constraint', build_constraints], 'CosyVoice 의존성 설치')\n"
    "        run([PYTHON, '-m', 'pip', 'check'], '의존성 충돌 확인')\n"
    '        marker.write_text(stamp)\n'
    '    download = (\n'
    "        'from huggingface_hub import snapshot_download; '\n"
    '        f\'snapshot_download("FunAudioLLM/CosyVoice2-0.5B", revision={MODEL_REVISION!r}, \'\n'
    '        f\'local_dir={str(MODEL)!r}, allow_patterns=["cosyvoice2.yaml", "llm.pt", "flow.pt", \'\n'
    '        \'"hift.pt", "campplus.onnx", "speech_tokenizer_v2.onnx", "CosyVoice-BlankEN/*"])\'\n'
    '    )\n'
    "    run([PYTHON, '-c', download], 'CosyVoice 2 모델 준비 (첫 실행 시 다운로드)')\n"
    "    missing = [name for name in ('cosyvoice2.yaml', 'llm.pt', 'flow.pt', 'hift.pt',\n"
    "                                'campplus.onnx', 'speech_tokenizer_v2.onnx') if not (MODEL / name).is_file()]\n"
    '    if missing:\n'
    "        raise RuntimeError('모델 다운로드가 완전하지 않습니다: ' + ', '.join(missing))\n"
    '    env = os.environ.copy()\n'
    '    # The isolated worker renders no notebook plots and may not have matplotlib_inline installed.\n'
    "    env['MPLBACKEND'] = 'Agg'\n"
    "    env['PYTHONPATH'] = os.pathsep.join([str(SOURCE), str(SOURCE / 'third_party/Matcha-TTS')])\n"
    "    libs = [str(p) for p in (ROOT / 'venv/lib/python3.10/site-packages/nvidia').glob('*/lib')]\n"
    "    env['LD_LIBRARY_PATH'] = os.pathsep.join(libs + [env.get('LD_LIBRARY_PATH', '')])\n"
    "    env['TOKENIZERS_PARALLELISM'] = 'false'\n"
    "    env['COSY_ACCESS_TOKEN'] = secrets.token_urlsafe(24)\n"
    '    with socket.socket() as sock:\n'
    "        sock.bind(('127.0.0.1', 0))\n"
    '        port = sock.getsockname()[1]\n'
    '    base = f\'http://127.0.0.1:{port}/v1/{env["COSY_ACCESS_TOKEN"]}\'\n'
    "    log_path = ROOT / 'server.log'\n"
    "    with log_path.open('w') as log:\n"
    "        worker = subprocess.Popen([str(PYTHON), '-u', str(Path(__file__).resolve()), '--serve', '--engine', ENGINE, '--port', str(port)],\n"
    '                                  cwd=SOURCE, env=env, stdout=log, stderr=subprocess.STDOUT)\n'
    "    print('\\n▶ 모델 준비 기록을 아래에 실시간으로 표시합니다. 최대 10분 후에도 준비되지 않으면 중단합니다.', flush=True)\n"
    '    ready = False\n'
    '    started = time.monotonic()\n'
    '    next_notice = started + 30\n'
    "    token = env['COSY_ACCESS_TOKEN']\n"
    "    live_log = log_path.open(encoding='utf-8', errors='replace')\n"
    '\n'
    '    def show_new_logs():\n'
    '        output = live_log.read()\n'
    '        if output:\n'
    "            print(output.replace(token, '[접속 키 숨김]'), end='' if output.endswith('\\n') else '\\n', flush=True)\n"
    '\n'
    '    def error_tail():\n'
    "        return log_path.read_text(errors='replace')[-7000:].replace(token, '[접속 키 숨김]')\n"
    '\n'
    '    try:\n'
    '        while time.monotonic() - started < 600:\n'
    '            show_new_logs()\n'
    '            if worker.poll() is not None:\n'
    "                raise RuntimeError('모델 시작 실패:\\n' + error_tail())\n"
    '            try:\n'
    '                ready = health(base)\n'
    '            except Exception:\n'
    '                pass\n'
    '            if ready:\n'
    "                state = {'base': base, 'pid': worker.pid, 'port': port, 'model': LABEL, 'engine': ENGINE}\n"
    '                STATE.write_text(json.dumps(state))\n'
    '                STATE.chmod(0o600)\n'
    '                show_new_logs()\n'
    "                print(f'✅ {LABEL} v{SERVER_VERSION} 준비 완료. 사이트 연결은 4번을 실행하세요.', flush=True)\n"
    '                return\n'
    '            now = time.monotonic()\n'
    '            if now >= next_notice:\n'
    '                elapsed = int(now - started)\n'
    "                print(f'  ⏳ 준비 대기 {elapsed // 60}분 {elapsed % 60:02d}초 / 최대 10분 — 아직 준비 완료가 아닙니다. 위 마지막 단계와 기록을 확인해주세요.', flush=True)\n"
    '                next_notice = now + 30\n'
    '            time.sleep(2)\n'
    '        show_new_logs()\n'
    "        raise RuntimeError('모델 준비가 10분 안에 완료되지 않아 중단했습니다. 아래 실제 기록을 보내주세요:\\n' + error_tail())\n"
    '    finally:\n'
    '        live_log.close()\n'
    '        if not ready and worker.poll() is None:\n'
    '            worker.terminate()\n'
    '\n'
    '\n'
    'def serve(port):\n'
    '    import faulthandler\n'
    '    # A live process is not proof of progress. Expose where startup is waiting.\n'
    '    faulthandler.enable()\n'
    '    faulthandler.dump_traceback_later(120, repeat=True)\n'
    "    print('[모델 준비 1/4] 오디오·Torch 실행 환경을 불러옵니다.', flush=True)\n"
    '    import numpy as np\n'
    '    import soundfile as sf\n'
    '    import torch\n'
    '    from fastapi import FastAPI, File, Form, HTTPException, UploadFile\n'
    '    from fastapi.responses import Response\n'
    '    import uvicorn\n'
    '    import fcntl\n'
    '    if not torch.cuda.is_available():\n'
    "        raise RuntimeError('CUDA GPU를 사용할 수 없습니다. T4 GPU 런타임인지 확인해주세요.')\n"
    "    print('[모델 준비 2/4] GPU 확인 완료. CosyVoice 실행 코드를 불러옵니다.', flush=True)\n"
    '    from cosyvoice.cli.cosyvoice import CosyVoice2\n'
    "    print('[모델 준비 3/4] 음성 모델·토크나이저를 불러옵니다. 세부 기록이 이어집니다.', flush=True)\n"
    '    model = CosyVoice2(model_dir=str(MODEL), load_jit=False, load_trt=False, fp16=False)\n'
    '    faulthandler.cancel_dump_traceback_later()\n'
    "    print('[모델 준비 4/4] 모델 로딩 완료. 연결 서버를 시작합니다.', flush=True)\n"
    '    sample_rate = model.sample_rate\n'
    "    access_token = os.environ['COSY_ACCESS_TOKEN']\n"
    '    model_lock = threading.Lock()\n'
    '    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)\n'
    '\n'
    '    def authorize(token):\n'
    '        if not secrets.compare_digest(token, access_token):\n'
    "            raise HTTPException(403, '연결 주소가 올바르지 않습니다. 새 주소 전체를 복사해주세요.')\n"
    '\n'
    "    @app.get('/')\n"
    '    def index():\n'
    "        return {'service': SERVICE, 'message': LABEL + ' 실행 중. 코랩에 표시된 전체 연결 주소를 사용하세요.'}\n"
    '\n'
    "    @app.get('/v1/{token}')\n"
    "    @app.get('/v1/{token}/health')\n"
    '    def status(token: str):\n'
    '        authorize(token)\n'
    "        return {'service': SERVICE, 'api_version': 1, 'ready': True, 'model': LABEL, 'engine': ENGINE, 'cuda': True,\n"
    "                'server_version': SERVER_VERSION, 'capabilities': ['style_instruction']}\n"
    '\n'
    "    @app.post('/v1/{token}/synthesize')\n"
    "    def synthesize(token: str, text: str = Form(...), prompt_text: str = Form(''),\n"
    '                   speed: float = Form(1.0), reference: UploadFile = File(...),\n'
    "                   style_instruction: str = Form('')):\n"
    '        authorize(token)\n'
    '        if not text.strip():\n'
    "            raise HTTPException(422, '생성할 대사를 입력해주세요.')\n"
    '        if not prompt_text.strip():\n'
    "            raise HTTPException(422, 'CosyVoice는 참조 음성의 실제 대사도 입력해야 합니다.')\n"
    '        if len(text) > 2000:\n'
    "            raise HTTPException(422, '한 번에 2,000자 이하로 나눠 생성해주세요.')\n"
    '        if not np.isfinite(speed) or not 0.5 <= speed <= 2:\n'
    "            raise HTTPException(422, '속도는 0.5~2.0 사이여야 합니다.')\n"
    '        style_instruction = style_instruction.strip()\n'
    "        if len(style_instruction) > 800 or '<|' in style_instruction or '|>' in style_instruction:\n"
    "            raise HTTPException(422, '스타일 지시문 형식이 올바르지 않습니다.')\n"
    '        if not model_lock.acquire(blocking=False):\n'
    "            raise HTTPException(409, '다른 음성을 생성 중입니다. 완료 후 다시 실행해주세요.')\n"
    '        gpu_lock = None\n'
    '        try:\n'
    "            gpu_lock = (ROOT.parent / 'gpu.lock').open('a+')\n"
    '            try:\n'
    '                fcntl.flock(gpu_lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n'
    '            except BlockingIOError:\n'
    "                raise HTTPException(409, '다른 엔진이 음성을 생성 중입니다. 완료 후 실행해주세요.')\n"
    '            payload = reference.file.read(10 * 1024 * 1024 + 1)\n'
    '            if not payload or len(payload) > 10 * 1024 * 1024:\n'
    "                raise HTTPException(422, '참조 음성은 10MB 이하 WAV 또는 MP3를 사용해주세요.')\n"
    "            with tempfile.TemporaryDirectory(prefix='cosy_ref_') as folder:\n"
    "                original = Path(folder) / 'reference.audio'\n"
    '                original.write_bytes(payload)\n'
    "                prepared = Path(folder) / 'reference.wav'\n"
    "                result = subprocess.run(['ffmpeg', '-nostdin', '-y', '-v', 'error', '-i', str(original),\n"
    "                                         '-t', '31', '-ac', '1', '-ar', '24000', str(prepared)],\n"
    '                                        capture_output=True, text=True, timeout=60)\n'
    '                if result.returncode:\n'
    "                    raise HTTPException(422, '참조 오디오를 읽을 수 없습니다. WAV 또는 MP3를 확인해주세요.')\n"
    "                audio, sr = sf.read(prepared, dtype='float32')\n"
    '                duration = len(audio) / sr\n'
    '                if not 3 <= duration <= 30:\n'
    "                    raise HTTPException(422, f'참조 음성은 3~30초여야 합니다. 현재 {duration:.1f}초입니다.')\n"
    '                if not np.all(np.isfinite(audio)) or np.max(np.abs(audio)) < 0.00001:\n'
    "                    raise HTTPException(422, '참조 음성에 들리는 목소리가 없습니다.')\n"
    '                pieces = []\n'
    "                sentences = re.split(r'(?<=[.!?。！？])\\s+|\\n+', text.strip())\n"
    '                chunks = []\n'
    '                for sentence in sentences:\n'
    '                    sentence = sentence.strip()\n'
    '                    while len(sentence) > 180:\n'
    "                        cut = sentence.rfind(' ', 60, 180)\n"
    '                        cut = cut if cut >= 60 else 180\n'
    '                        chunks.append(sentence[:cut])\n'
    '                        sentence = sentence[cut:].strip()\n'
    '                    if sentence:\n'
    '                        chunks.append(sentence)\n'
    '                with torch.inference_mode():\n'
    '                    for chunk in chunks:\n'
    '                        if style_instruction:\n'
    '                            # The pinned upstream example adds this delimiter at the caller.\n'
    '                            # Instruction and reference transcript must never be concatenated.\n'
    "                            instruction = 'Speak in Korean. ' + style_instruction + '<|endofprompt|>'\n"
    '                            generated = model.inference_instruct2(chunk, instruction, str(prepared),\n'
    '                                                                 stream=False, speed=speed, text_frontend=False)\n'
    '                        else:\n'
    '                            generated = model.inference_zero_shot(chunk, prompt_text.strip(), str(prepared),\n'
    '                                                                  stream=False, speed=speed, text_frontend=False)\n'
    '                        for item in generated:\n'
    "                            pieces.append(item['tts_speech'].detach().cpu().numpy().reshape(-1))\n"
    '                if not pieces or sum(x.size for x in pieces) == 0:\n'
    "                    raise RuntimeError('모델이 빈 음성을 반환했습니다. 참조 음성과 실제 대사를 확인해주세요.')\n"
    '                speech = np.concatenate(pieces)\n'
    '                if not np.all(np.isfinite(speech)) or np.max(np.abs(speech)) < 0.000001:\n'
    "                    raise RuntimeError('모델 출력이 무음이거나 손상되었습니다. server.log를 확인해주세요.')\n"
    '                output = io.BytesIO()\n'
    "                sf.write(output, speech, sample_rate, format='WAV', subtype='PCM_16')\n"
    "                return Response(output.getvalue(), media_type='audio/wav')\n"
    '        except HTTPException:\n'
    '            raise\n'
    '        except Exception as exc:\n'
    "            logging.exception('%s synthesis failed', LABEL)\n"
    "            raise HTTPException(500, f'{type(exc).__name__}: {str(exc)[:600]}') from exc\n"
    '        finally:\n'
    '            reference.file.close()\n'
    '            if gpu_lock is not None:\n'
    '                gpu_lock.close()\n'
    '            model_lock.release()\n'
    '\n'
    "    uvicorn.run(app, host='127.0.0.1', port=port, access_log=False)\n"
    '\n'
    '\n'
    'def tunnel():\n'
    '    state = read_state()\n'
    "    print('음성 서버 준비 상태를 확인합니다…', flush=True)\n"
    '    try:\n'
    "        ready = bool(state.get('base')) and health(state['base'])\n"
    '    except Exception:\n'
    '        ready = False\n'
    '    if not ready:\n'
    "        raise RuntimeError('음성 서버가 아직 준비되지 않았거나 중지됐습니다. 1번 셀에서 준비 완료 메시지가 나온 뒤 4번을 실행해주세요.')\n"
    "    if state.get('public_base'):\n"
    '        try:\n'
    "            if health(state['public_base']):\n"
    "                print(LABEL + ' 프로그램 연결 주소:\\n' + state['public_base'], flush=True)\n"
    '                return\n'
    '        except Exception:\n'
    '            pass\n'
    "    executable = ROOT / 'cloudflared'\n"
    '    if not executable.is_file():\n'
    "        print('연결 도구 다운로드 중…', flush=True)\n"
    "        temp = executable.with_suffix('.download')\n"
    "        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', temp)\n"
    '        temp.chmod(0o755)\n'
    '        temp.replace(executable)\n'
    "    log_path = ROOT / 'tunnel.log'\n"
    "    print('외부 연결을 여는 중입니다. 연결 주소가 확인될 때까지 기다려주세요 (최대 약 2분).', flush=True)\n"
    "    with log_path.open('w') as log:\n"
    "        proc = subprocess.Popen([str(executable), 'tunnel', '--no-autoupdate', '--url',\n"
    '                                 f\'http://127.0.0.1:{state["port"]}\'], stdout=log, stderr=subprocess.STDOUT)\n'
    '    success = False\n'
    '    try:\n'
    '        deadline = time.monotonic() + 120\n'
    '        next_notice = time.monotonic() + 15\n'
    '        while time.monotonic() < deadline:\n'
    '            if proc.poll() is not None:\n'
    "                raise RuntimeError('연결 도구가 종료되었습니다:\\n' + log_path.read_text(errors='replace')[-2000:])\n"
    "            match = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', log_path.read_text(errors='replace'))\n"
    '            if match:\n'
    "                public = match.group(0) + state['base'].split(str(state['port']), 1)[1]\n"
    '                try:\n'
    '                    if health(public):\n'
    '                        state.update(public_base=public, tunnel_pid=proc.pid)\n'
    '                        STATE.write_text(json.dumps(state))\n'
    "                        print('\\n✅ ' + LABEL + ' 프로그램 연결 준비 완료\\n프로그램 연결 주소:\\n' + public, flush=True)\n"
    "                        print(f'위 주소 전체를 AI Voice Studio의 {LABEL} 접속 주소에 붙여 넣으세요.', flush=True)\n"
    '                        success = True\n'
    '                        return\n'
    '                except Exception:\n'
    '                    pass\n'
    '            if time.monotonic() >= next_notice:\n'
    "                print('연결 주소를 확인하는 중… 아직 연결 완료가 아닙니다.', flush=True)\n"
    '                next_notice = time.monotonic() + 15\n'
    '            time.sleep(2)\n'
    "        raise RuntimeError('외부 연결을 열지 못했습니다. 코랩 내부의 2~3번 셀은 계속 사용할 수 있습니다.')\n"
    '    finally:\n'
    '        if not success and proc.poll() is None:\n'
    '            proc.terminate()\n'
    '\n'
    '\n'
    'def main():\n'
    '    parser = argparse.ArgumentParser()\n'
    '    modes = parser.add_mutually_exclusive_group(required=True)\n'
    "    modes.add_argument('--setup', action='store_true')\n"
    "    modes.add_argument('--serve', action='store_true')\n"
    "    modes.add_argument('--tunnel', action='store_true')\n"
    "    parser.add_argument('--port', type=int, default=50000)\n"
    "    parser.add_argument('--engine', choices=['cosyvoice'], default='cosyvoice')\n"
    '    args = parser.parse_args()\n'
    '    if args.setup:\n'
    '        try:\n'
    '            setup()\n'
    '        except Exception:\n'
    '            if ROOT.is_dir():\n'
    "                with (ROOT / 'setup.log').open('a', encoding='utf-8') as log:\n"
    '                    traceback.print_exc(file=log)\n'
    '            raise\n'
    '    elif args.serve:\n'
    '        serve(args.port)\n'
    '    else:\n'
    '        tunnel()\n'
    '\n'
    '\n'
    "if __name__ == '__main__':\n"
    '    main()\n'
)
server_script = Path('/content/ai_voice_cosy_runner.py')
server_script.write_text(server_source, encoding='utf-8')
log_dir = Path('/content/ai_voice_dual_v1/cosyvoice')
log_dir.mkdir(parents=True, exist_ok=True)
recent = deque(maxlen=55)
with (log_dir / 'notebook.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen([sys.executable, '-u', str(server_script), '--setup'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            recent.append(line)
        code = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
if code:
    print('\n【설치가 멈춘 실제 오류 · 이 부분을 보내주세요】\n' + ''.join(recent), flush=True)
    raise RuntimeError('설치가 중단되었습니다. 위 【실제 오류】 내용을 확인해주세요.') from None
print('✅ CosyVoice 2 준비 완료! 사이트 연결은 4번, 코랩에서 직접 만들기는 2·3번을 실행하세요.')


In [ ]:
#@title 2. 참고 음성 업로드 — 목소리를 바꿀 때 실행
from google.colab import files
from pathlib import Path
import hashlib
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('참고 음성 WAV 또는 MP3 파일 한 개를 선택해주세요.')
name, contents = next(iter(uploaded.items()))
if Path(name).suffix.lower() not in ('.wav', '.mp3'):
    raise ValueError('WAV 또는 MP3 파일을 선택해주세요.')
if len(contents) > 10 * 1024 * 1024:
    raise ValueError('3~30초, 10MB 이하의 음성을 선택해주세요.')
reference_dir = Path('/content/ai_voice_dual_v1/references')
reference_dir.mkdir(parents=True, exist_ok=True)
reference_path = reference_dir / (hashlib.sha256(contents).hexdigest()[:16] + Path(name).suffix.lower())
reference_path.write_bytes(contents)
print('참고 음성 준비 완료:', name)
print('이제 3번에서 참고 음성의 실제 대사와 새로 읽힐 대사를 입력하세요.')


In [ ]:
#@title 3. 대사 입력 → CosyVoice 음성 생성·다운로드
새로_읽힐_대사 = "옛날 어느 산골 마을에 마음씨 착한 청년이 살고 있었습니다." #@param {type:"string"}
참고음성_실제대사 = "" #@param {type:"string"}
말하기_속도 = 1.0 #@param {type:"number"}
import json
import io
import uuid
import wave
from pathlib import Path
import requests
from IPython.display import Audio, display
from google.colab import files
engine = 'cosyvoice'
prompt = 참고음성_실제대사.strip()
if not prompt:
    raise ValueError('참고 음성에서 실제로 말한 내용을 그대로 입력해주세요.')
if not 새로_읽힐_대사.strip():
    raise ValueError('새로 읽힐 대사를 입력해주세요.')
if 'reference_path' not in globals() or not reference_path.is_file():
    raise ValueError('먼저 2번에서 참고 음성을 업로드해주세요.')
state_file = Path('/content/ai_voice_dual_v1') / engine / 'state.json'
if not state_file.is_file():
    raise RuntimeError('CosyVoice가 준비되지 않았습니다. 먼저 1번을 실행해주세요.')
state = json.loads(state_file.read_text())
print('CosyVoice 음성 생성 중…')
try:
    with reference_path.open('rb') as audio:
        response = requests.post(state['base'] + '/synthesize',
            data={'text': 새로_읽힐_대사, 'prompt_text': prompt, 'speed': 말하기_속도},
            files={'reference': (reference_path.name, audio, 'application/octet-stream')},
            timeout=(10, 600))
except requests.RequestException as exc:
    raise RuntimeError('코랩 연결이 끊겼거나 응답 시간이 초과되었습니다. 1번 준비 상태와 마지막 로그를 확인해주세요.') from exc
if not response.ok:
    try:
        message = response.json().get('detail', response.text[:400])
    except ValueError:
        message = response.text[:400]
    raise RuntimeError(f'음성 생성 실패 ({response.status_code}): {message}')
with wave.open(io.BytesIO(response.content), 'rb') as wav:
    if wav.getnframes() == 0:
        raise RuntimeError('빈 음성이 반환되었습니다. 아래 오류 확인 셀을 실행해주세요.')
output_dir = Path('/content/ai_voice_dual_v1/outputs')
output_dir.mkdir(exist_ok=True)
output = output_dir / (engine + '_' + uuid.uuid4().hex[:8] + '.wav')
output.write_bytes(response.content)
display(Audio(filename=str(output)))
print('생성 완료. 음성을 듣고 다운로드하세요.')
files.download(str(output))


## 선택: 공유 사이트의 AI Voice Studio에 연결

1번에 **준비 완료**가 나온 뒤 아래 **4번 왼쪽 ▶**만 누르세요.
1/3 → 2/3 → 3/3 진행 메시지와 연결 주소가 표시됩니다. 체크박스는 없습니다.
주소가 표시된 뒤 실행 버튼이 ▶로 돌아오는 것은 정상입니다.
나온 **프로그램 연결 주소 전체**를 사이트 왼쪽 **CosyVoice 2 · 내 코랩 주소** 칸에 붙여 넣으세요.
주소 뒤의 `/v1/…` 부분도 포함합니다.

주소에는 임시 접속 키가 포함됩니다. 본인만 사용하고, 런타임 종료 후에는 새 주소를 받으세요.
무료 코랩에서는 외부 웹 화면을 통한 사용이 제한되어 세션이 종료될 수 있습니다.
연결이 안 되면 코랩 2·3번에서 직접 생성할 수 있습니다. [Colab 안내](https://research.google.com/colaboratory/faq.html)


In [ ]:
#@title 4. 사이트 연결 · 왼쪽 ▶만 누르세요 (v2.9.2)
from pathlib import Path
import hashlib
import runpy
import urllib.request

print("1/3 연결 실행 파일 확인 중…", flush=True)
RUNNER = Path("/content/ai_voice_cosy_runner.py")
EXPECTED = "d477f647f8953a778325f21f975b76b80c9a48a916b1284484ed4ec34e047339"
runner_bytes = RUNNER.read_bytes() if RUNNER.is_file() else b""
if hashlib.sha256(runner_bytes).hexdigest() != EXPECTED:
    print("최신 연결 실행 파일로 복구 중…", flush=True)
    revision = "648420f734f165f7e6446e20c9dc57d3bc384326"
    source_url = ("https://raw.githubusercontent.com/ssss2513-cyber/"
                  f"ai-audio-studio/{revision}/colab_server.py")
    try:
        with urllib.request.urlopen(source_url, timeout=30) as response:
            runner_bytes = response.read(1024 * 1024)
    except Exception as exc:
        raise RuntimeError("실행 파일을 가져오지 못했습니다. 코랩 인터넷 연결을 확인한 뒤 4번을 다시 실행해주세요.") from exc
    if hashlib.sha256(runner_bytes).hexdigest() != EXPECTED:
        raise RuntimeError("실행 파일 확인에 실패했습니다. 새 코랩을 열어 다시 실행해주세요.")
    RUNNER.write_bytes(runner_bytes)

runner_api = runpy.run_path(str(RUNNER))
print("2/3 음성 서버 준비 상태 확인 중…", flush=True)
state = runner_api["read_state"]()
try:
    ready = bool(state.get("base")) and runner_api["health"](state["base"])
except Exception:
    ready = False
if not ready:
    raise RuntimeError("1번의 음성 서버가 아직 준비되지 않았거나 중지됐습니다. 1번에서 ‘준비 완료’가 나온 뒤 4번을 다시 실행해주세요. 1번이 계속 실행 중이면 함께 실행하지 말고 먼저 끝날 때까지 기다려주세요.")

print("3/3 사이트 연결 주소 생성 중…", flush=True)
runner_api["tunnel"]()
state = runner_api["read_state"]()
address = state.get("public_base")
try:
    connected = bool(address) and runner_api["health"](address)
except Exception:
    connected = False
if not connected:
    raise RuntimeError("외부 연결이 확인되지 않았습니다. 아래 오류 확인 셀의 마지막 기록을 확인해주세요.")

print("\n✅ 연결 확인 완료! 아래 주소 전체를 사이트의 CosyVoice 2 · 내 코랩 주소 칸에 붙여넣으세요.", flush=True)
print(address, flush=True)
print("실행 버튼이 ▶로 돌아와도 서버는 계속 실행됩니다. 사용 중에는 코랩 런타임을 유지하세요.", flush=True)


In [ ]:
#@title 오류가 날 때만 실행 · 출력 내용을 보내주세요
from pathlib import Path
for name in ('notebook.log', 'setup.log', 'server.log', 'tunnel.log'):
    log = Path('/content/ai_voice_dual_v1/cosyvoice') / name
    print('\n【' + name + '】')
    print('\n'.join(log.read_text(errors='replace').splitlines()[-45:]) if log.is_file() else '아직 기록 없음')


### 실행 방식

CosyVoice는 별도의 Python 3.10 환경에서 실행합니다. 설치 오류가 나면 중지하며 모델 로딩이 끝난 뒤에 준비 완료를 표시합니다.
음성 생성이 실패해도 같은 요청을 자동으로 반복하지 않습니다.

현재 버전의 실제 Colab GPU 음성 생성은 별도 확인이 필요합니다.
공식 자료: [CosyVoice](https://github.com/FunAudioLLM/CosyVoice).
